# 拓展：数据工程思维——从数据流转到数据质量工程

> **关联主课**：W4D1 数据从哪来——从业务动作到数据表
> **难度**：⭐⭐⭐⭐ | **新增概念**：8个课程未覆盖的深度概念
> **建议时间**：2-2.5小时 | **深度配比**：代码深度 7/8，案例深度 1/8
> **前置**：完成W4D1主课学习 + 至少用过一张数仓表写SQL

## 📡 知识网络预览

```
                        ┌──────────────────────────────┐
                        │    W4D1 核心：数据流转认知     │
                        │  OLTP→采集→Kafka→ETL→数仓→应用 │
                        └──────────────┬───────────────┘
                                       │
            ┌──────────────────────────┼──────────────────────────┐
            │                          │                          │
            ▼                          ▼                          ▼
   ┌────────────────┐      ┌──────────────────┐      ┌──────────────────┐
   │ [前置] CDC实战  │      │ [防护] 数据质量三件套│      │ [替代] 架构演进决策│
   │ INSERT/UPDATE/ │      │ Pandera+GE+漂移   │      │ ETL→ELT→Zero·ETL │
   │ DELETE变更捕获  │      │ 运行时Schema验证   │      │ Lambda vs Kappa  │
   └───────┬────────┘      └────────┬─────────┘      └────────┬─────────┘
           │                        │                          │
           └────────────────────────┼──────────────────────────┘
                                    │
                    ┌───────────────┼───────────────┐
                    │               │               │
                    ▼               ▼               ▼
          ┌─────────────┐  ┌──────────────┐  ┌──────────────┐
          │[落地] 血缘追踪│  │[防护] SQL反模式│  │[落地] SLA监控 │
          │SQL→DAG生成  │  │自动检测器     │  │延迟自动告警   │
          └─────────────┘  └──────────────┘  └──────────────┘
```

**本拓展新增 8 个课程未覆盖的深度概念：**

| # | 概念群 | 关系类型 | 深度类型 | 来源 | 必修/选修 |
|---|--------|---------|---------|------|----------|
| 1 | CDC（Change Data Capture）实战模拟 | [前置] | 代码 | 主课浅层→深层实现 | 必修 |
| 2 | Pandera 运行时Schema验证 | [防护] | 代码 | 永久空白区 P1 | 必修 |
| 3 | Great Expectations 数据质量自动化 | [防护] | 代码 | 永久空白区 P1 | 必修 |
| 4 | 数据漂移检测（Evidently + 统计检验） | [防护] | 代码 | 永久空白区 P1 | 必修 |
| 5 | SQL反模式自动检测器 | [防护] | 代码 | 永久空白区 P1·SQL反模式 | 必修 |
| 6 | 数据血缘自动追踪（SQL→DAG） | [落地] | 代码 | 主课概念→工程实现 | 选修 |
| 7 | 数据延迟SLA监控自动化 | [落地] | 代码 | 主课手动清单→自动系统 | 必修 |
| 8 | 数据架构演进：ETL→ELT→Zero-ETL & Lambda/Kappa | [替代] | 案例 | 主课ETLvsELT→完整演进 | 选修 |

**阅读指引**：
- 🟢 **全读**：先做入口自诊（Part 2），判断基础 → 按顺序读概念群 1-8
- 🟡 **选读**：如果你已经用过Pandera/GE，可以跳到概念群 5（SQL反模式）
- 🔴 **重点关注**：概念群 2-4（数据质量三件套）是永久空白区最高优先级内容

# 第一部分：沙盘设定

## 🏢 公司画像

**快达出行 (Kuaida)** — 国内前三的出行平台，覆盖网约车、顺风车、企业用车三条业务线。

- **规模**：DAU 2000万，日均订单 800万单，入驻司机 120万
- **商业模式**：撮合乘客与司机，抽佣 20-25%。核心收入 = (客单价 - 司机分成) × 订单量
- **技术栈**：MySQL 业务库(下单/支付/用户) → Canal → Kafka → Hive/Spark 数仓 → Metabase/自研BI
- **数据团队**：从5人扩张到20人，新增了数据质量、数据治理两条职能线

## 🎯 当前挑战

快达出行的数据团队正在经历"成长的烦恼"：

1. **数据质量失控**：团队扩张后，不同分析师写SQL时对同一指标算出不同结果。"GMV"有4种口径，没人说得清哪个是对的
2. **延迟排查靠人**：每天早上运营在群里催数据，分析师手动查调度系统，"像消防员一样到处救火"
3. **新人上手难**：新来的分析师面对 300+ 张表无从下手，不知道哪张可信、哪张有坑
4. **数据事故频发**：上个月一次ETL任务修改导致下游 12 张报表全错，花了3天才修复——因为没有数据血缘追踪

## 👤 你的角色

你是快达出行数据团队的 **高级数据分析师 (Senior Data Analyst)**。

你的职责不只是"写SQL"——你是数据基础设施的"守门人"。你需要在团队扩张期建立数据质量体系，让数据变得**可信、可追溯、可监控**。

你的直接利益相关方：
- **运营总监 王姐**：每天早上9点要看日报，延迟超过10分钟会直接找CTO
- **数仓工程师 老李**：负责ETL任务，但讨厌被打断问"数据为什么还没出来"
- **新来的分析师 小陈**：入职2周，每天问你"这张表什么意思"
- **CTO 张总**：关心数据事故为什么频发，要求你给出解决方案

## 💾 数据资产

你在快达出行日常接触的核心数据表：

**业务库（MySQL - OLTP）**：
| 表名 | 关键字段 | 日增量 |
|------|---------|--------|
| orders | order_id, user_id, driver_id, amount, status, created_at | ~800万行 |
| users | user_id, phone, city, register_time | ~50万行 |
| trips | trip_id, order_id, start_time, end_time, distance, fare | ~800万行 |
| payments | payment_id, order_id, method, amount, status | ~800万行 |

**数仓（Hive - OLAP）**：
| 表名 | 层级 | 粒度 | 时效 | 说明 |
|------|------|------|------|------|
| ods_order_log_df | ODS | 行 | T+1 03:00 | 订单binlog原始快照 |
| ods_user_info_df | ODS | 行 | T+1 03:00 | 用户信息全量快照 |
| dwd_order_detail_df | DWD | 行 | T+1 05:00 | 清洗后订单明细 |
| dwd_trip_fact_df | DWD | 行 | T+1 05:00 | 行程事实表 |
| dws_user_daily_1d | DWS | 用户×天 | T+1 06:00 | 用户日活汇总 |
| dws_order_daily_1d | DWS | 天 | T+1 06:00 | 订单日汇总 |
| ads_ceo_dashboard | ADS | 天 | T+1 08:00 | CEO看板 |
| ads_operation_daily | ADS | 天 | T+1 08:00 | 运营日报 |
| dim_user | 维表 | 用户 | T+1 04:00 | 用户画像维表 |
| dim_driver | 维表 | 司机 | T+1 04:00 | 司机画像维表 |

**Kafka Topics**：
- `order_events` — 订单事件流（下单/支付/取消/退款）
- `trip_events` — 行程事件流（开始/结束/评价）
- `user_events` — 用户行为事件流（登录/注册/领券）

> ⚠️ **关键约束**：以上所有数据都"模拟存在"——本拓展中的所有代码示例都基于这些表和字段。你不需要真正连接数据库，所有练习使用 Python 模拟数据即可完成。

# 第二部分：入口自诊
> ⏱ 预计 5 分钟 | 只测主课程内容 | 共 5 题

## 📋 自诊题

**题1**：`dws_user_daily_1d` 表的数据在每天凌晨 6:00 产出。今天早上 9:00 你发现昨天的分区数据为空。你应该从哪一层开始排查？
A. 直接查 `ods_order_log_df`（最底层）
B. 先确认看板/BI配置是否正确（最上层）
C. 查 `dwd_order_detail_df` 的分区是否存在（中间层）
D. 重启ETL任务

**题2**：OLTP和OLAP必须分离的第一性原理是什么？
A. OLTP太贵，OLAP便宜
B. 实时读写的OLTP和批量扫描的OLAP对资源的需求是冲突的——分析SQL会拖垮业务
C. 法律规定生产数据和历史数据必须分开存储
D. OLTP不支持SQL

**题3**：运营的DAU(102万)和App后台的在线用户峰值(98万)差了4万。最可能的原因是什么？
A. 数据采集丢失了4万用户
B. 数仓ETL任务有bug
C. 口径不同：DAU是累计去重用户数，后台是在线设备瞬时峰值
D. App后台数据不准

**题4**：Kafka在数据架构中的三个核心作用是什么？（写出关键词即可）

**题5**：`ods_`、`dwd_`、`dws_`、`ads_` 四层分别代表什么？简述每层的职责。

## ✅ 自诊反馈

**题1答案**：**B → C → A**（从上往下确认表现，从下往上游查根因）

| ❓ 常见偏差 | 🔧 补救路径 |
|-----------|------------|
| 直接查ODS层（跳过中间层） | 浪费时间——如果ADS层根本没被调度，查ODS是白费功夫 |
| 只查DWD不查调度 | 调度系统可能根本就没触发——先看调度状态，再看分区 |

> 如果你选了D（重启ETL），说明你需要重读主课区块3的步骤2——"先定位再修复"。

**题2答案**：**B**。关键句：分析SQL扫描千万行需要大量CPU和IO，会拖慢INSERT——用户下单卡顿→流失。

**题3答案**：**C**。80%的"数据对不上"是口径不同。

**题4答案**：削峰填谷、解耦生产者和消费者、支持多订阅。缺一不可。

**题5答案**：ods_=操作数据层(原始数据照搬业务库), dwd_=明细数据层(清洗标准化), dws_=汇总数据层(轻度聚合), ads_=应用数据层(面向看板/报表)。

### 📊 自诊结果评估
| 得分 | 诊断 | 建议 |
|------|------|------|
| 5/5 | ✅ 主课掌握扎实 | 直接进入Part 3概念网络 |
| 3-4/5 | ⚠️ 主课理解有盲区 | 先重读主课对应区块，再进入拓展 |
| 0-2/5 | 🔴 建议重学主课 | 拓展的深度概念建立在你理解数据流转链路的基础上 |

# 第三部分：概念网络展开
---
## 🔬 概念群 1：CDC（Change Data Capture）实战模拟

> **关系类型**：[前置] — 主课讲"Canal监听binlog"，但没讲Canal到底在做什么。理解CDC是理解"数据如何从OLTP到OLAP"的工程基础。
> **[必修]** | 代码深度 | 预计 15 分钟

### 🔴 认知冲突

运营王姐发来一个问题：

> "我每天下午3点看 `dws_user_daily_1d` 表里昨天的DNU（日新增用户数）是 12,564。但我刚刚从MySQL `users` 表直接查昨天注册的用户数，结果是 12,568。差了4个人。这4个人去哪了？"

**请用主课知识尝试回答**：你首先想到的原因是什么？你能确定是"采集时漏了"还是"数仓多记了"吗？

*思考30秒再往下翻。*

---

你可能想到了"口径不同"——但这里口径完全一致（都是昨天注册的用户数）。你也可能想到了"ETL延迟"——但现在是下午3点，ETL早就跑完了。

**真正的答案隐藏在CDC的细节里**：那4个用户确实注册了——但他们在注册后**立刻注销了账号**（在MySQL中执行了DELETE）。Canal同步了INSERT事件到Kafka，数仓也收到了这4条记录。但后来Canal同步DELETE事件时，Kafka消费者恰好宕机了30秒——**DELETE事件丢失了**。所以数仓里有这4个用户，MySQL里没有。

**这不是口径问题，这是CDC的数据一致性问题。**

### 📘 概念引入：CDC（Change Data Capture）

#### ① 一句话白话
"CDC就是把数据库的每一次增删改操作记录下来，像会计记账一样——不是只记'现在的余额'，而是记'每一笔流水'。"

#### ② 核心逻辑 + 具体例子

CDC有三种实现方式：
- **方式1：Log-based CDC（Canal/Flink CDC）**：读取MySQL binlog，捕获INSERT/UPDATE/DELETE事件。优点：实时、低影响；缺点：只能捕获有主键的表
- **方式2：Query-based CDC（定时SELECT）**：每小时跑一次`SELECT * WHERE updated_at >= ...`。优点：简单；缺点：无法捕获DELETE，UPDATE被"压缩"
- **方式3：Trigger-based CDC**：在表上创建触发器。优点：完整；缺点：拖慢业务库

#### ③ 与主课核心概念的关系
主课讲的"Canal监听MySQL binlog"就是方式1（Log-based CDC）。但我们只讲了"它能做到"，没讲"它可能在哪个环节出问题"。这个拓展把你的认知从"知道有CDC"升级到"理解CDC的一致性边界"。

| CDC方式 | 能否捕获DELETE | 对业务库影响 | 主流工具 |
|---------|--------------|-------------|---------|
| Log-based | ✅（binlog有DELETE事件）| 几乎无 | Canal, Flink CDC |
| Query-based | ❌（已被物理删除）| 有（定时SELECT）| DataX, Sqoop |
| Trigger-based | ✅（触发器记录）| 有（同步触发）| 自建 |

### 🎯 在沙盘中的用武之地
快达出行的 `ods_order_log_df` 表。运营发现"数仓有12,568个新用户但MySQL只有12,564个"。
如果你理解了CDC：
- 你能立刻判断：大概率是DELETE事件在Kafka消费环节丢失了
- 你能通过对比 `ods_order_log_df` 中每种操作类型的数量来验证
- 你能给出修复时间估算，不会浪费时间排查"ETL JOIN逻辑"——因为问题在更上游

In [1]:
# ===== 练习1：实现一个迷你CDC模拟器 =====
# 练习目标：用Python模拟 MySQL binlog → CDC → 数仓 的完整流程
# 你需要实现:
#   1. 一个模拟的'业务库'(用字典存储当前状态)
#   2. 一个'binlog'(用列表记录所有变更事件)
#   3. 一个'数仓'(消费binlog事件,维护自己的副本)
#   4. 模拟'Kafka消费者宕机'导致部分事件丢失
#   5. 对比业务库和数仓的结果,检测一致性问题
import json
from dataclasses import dataclass
from typing import Dict, List, Optional
from datetime import datetime

# ===== STEP 1: 定义变更事件 =====
@dataclass
class ChangeEvent:
    event_id: int
    operation: str   # 'INSERT', 'UPDATE', 'DELETE'
    table: str
    row_id: int
    data: Dict       # INSERT/UPDATE时的数据
    timestamp: str

# ===== STEP 2: 模拟业务库(MySQL) =====
class BusinessDB:
    def __init__(self):
        self.tables = {'users': {}, 'orders': {}}
        self.binlog = []
        self.event_counter = 0

    def insert(self, table, row_id, data):
        self.tables[table][row_id] = data.copy()
        self.event_counter += 1
        event = ChangeEvent(
            event_id=self.event_counter, operation='INSERT',
            table=table, row_id=row_id, data=data.copy(),
            timestamp=datetime.now().isoformat())
        self.binlog.append(event)
        return event

    def delete(self, table, row_id):
        if row_id not in self.tables[table]:
            return None
        del self.tables[table][row_id]
        self.event_counter += 1
        event = ChangeEvent(
            event_id=self.event_counter, operation='DELETE',
            table=table, row_id=row_id, data={},
            timestamp=datetime.now().isoformat())
        self.binlog.append(event)
        return event

    def count(self, table):
        return len(self.tables[table])

# ===== STEP 3: 模拟数仓(Hive) =====
class DataWarehouse:
    def __init__(self):
        self.tables = {'users': {}, 'orders': {}}
        self.applied_events = []

    def apply_event(self, event):
        if event.event_id in self.applied_events:
            return False  # 幂等:已消费的不重复处理
        if event.operation in ('INSERT', 'UPDATE'):
            self.tables[event.table][event.row_id] = event.data.copy()
        elif event.operation == 'DELETE':
            self.tables[event.table].pop(event.row_id, None)
        self.applied_events.append(event.event_id)
        return True

    def count(self, table):
        return len(self.tables[table])

# ===== STEP 4: 模拟Kafka消费者(可能丢失事件) =====
class KafkaConsumer:
    def __init__(self, crash_range=None):
        self.crash_range = crash_range  # (start_id, end_id) 范围内的事件丢失

    def consume(self, binlog, warehouse):
        consumed = lost = 0
        for event in binlog:
            if self.crash_range and self.crash_range[0] <= event.event_id <= self.crash_range[1]:
                lost += 1
                continue  # 模拟丢失
            warehouse.apply_event(event)
            consumed += 1
        return {'consumed': consumed, 'lost': lost}

# ===== STEP 5: 完整模拟 =====
print('=' * 60)
print('CDC 模拟实验：数据一致性检测')
print('=' * 60)

db = BusinessDB()
dw = DataWarehouse()

# 模拟: 10个用户注册, 其中2个随即注销
for i in range(1, 11):
    db.insert('users', i, {'name': f'用户{i}', 'register_time': '2026-07-11 09:00:00'})
print(f'业务库: 插入了10个用户, users表行数 = {db.count("users")}')

# 用户9和10立即注销!
db.delete('users', 9)
db.delete('users', 10)
print(f'业务库: 用户9和10注销后, users表行数 = {db.count("users")}')

# 再注册2个用户
db.insert('users', 11, {'name': '用户11', 'register_time': '2026-07-11 10:00:00'})
db.insert('users', 12, {'name': '用户12', 'register_time': '2026-07-11 10:01:00'})

print(f'业务库最终状态: users行数 = {db.count("users")}')
print(f'binlog事件总数: {len(db.binlog)} (10 INSERT + 2 DELETE + 2 INSERT = 14)')

# 实验1: 正常消费
print('\n--- 消费者1: 正常消费(无故障) ---')
consumer_normal = KafkaConsumer(crash_range=None)
result = consumer_normal.consume(db.binlog, dw)
print(f'消费: {result["consumed"]}, 丢失: {result["lost"]}')
print(f'数仓users行数: {dw.count("users")}')
check1 = '一致' if db.count('users') == dw.count('users') else '不一致!'
print(f'一致性检查: {"✅ " + check1 if check1 == "一致" else "❌ " + check1}')

# 实验2: 消费者宕机, DELETE事件丢失!
dw2 = DataWarehouse()
# 用户9 DELETE的event_id=11, 用户10 DELETE的event_id=12 (前面10个INSERT是1-10)
consumer_broken = KafkaConsumer(crash_range=(11, 12))  # 2个DELETE事件丢失!
print('\n--- 消费者2: 故障消费(DELETE事件丢失) ---')
result2 = consumer_broken.consume(db.binlog, dw2)
print(f'消费: {result2["consumed"]}, 丢失: {result2["lost"]}')
print(f'数仓users行数: {dw2.count("users")}')
check2 = '一致' if db.count('users') == dw2.count('users') else '不一致!'
print(f'一致性检查: {"✅ " + check2 if check2 == "一致" else "❌ " + check2}')

# 诊断差异
mysql_users = set(db.tables['users'].keys())
hive_users = set(dw2.tables['users'].keys())
only_in_hive = hive_users - mysql_users
only_in_mysql = mysql_users - hive_users
print(f'\n🔍 差异诊断:')
print(f'  只在数仓存在的用户: {only_in_hive}')
print(f'  只在MySQL存在的用户: {only_in_mysql}')
print(f'  根因: DELETE事件丢失, 导致 {len(only_in_hive)} 个用户被错误保留')

print('\n' + '='*60)
print('结论: Log-based CDC不是完美无缺的——消费者故障会导致')
print('数据不一致。这就是为什么需要全量快照+增量日志双验证。')
print('='*60)

CDC 模拟实验：数据一致性检测
业务库: 插入了10个用户, users表行数 = 10
业务库: 用户9和10注销后, users表行数 = 8
业务库最终状态: users行数 = 10
binlog事件总数: 14 (10 INSERT + 2 DELETE + 2 INSERT = 14)

--- 消费者1: 正常消费(无故障) ---
消费: 14, 丢失: 0
数仓users行数: 10
一致性检查: ✅ 一致

--- 消费者2: 故障消费(DELETE事件丢失) ---
消费: 12, 丢失: 2
数仓users行数: 12
一致性检查: ❌ 不一致!

🔍 差异诊断:
  只在数仓存在的用户: {9, 10}
  只在MySQL存在的用户: set()
  根因: DELETE事件丢失, 导致 2 个用户被错误保留

结论: Log-based CDC不是完美无缺的——消费者故障会导致
数据不一致。这就是为什么需要全量快照+增量日志双验证。


### 🔧 练习反馈

| ✅ 参考答案 | ❓ 常见偏差 | 🔧 补救路径 |
|-----------|-----------|------------|
| 数仓中多了被"误复活"的用户(user_id=9,10)，因为DELETE丢失 | 以为CDC一定保证一致 | 理解at-least-once vs exactly-once语义——CDC通常只保证at-least-once |
| 根因是日志消费侧的故障窗口 | 归因于"ETL JOIN逻辑" | 数据流转问题要先判断在哪一层出现的差异——逐层对比 |
| 修复：重放binlog或跑全量快照 | "简单删掉就好了" | 不能手动修复——需要用binlog回放机制，保证可溯源 |

> 💡 **工程师视角**：CDC的世界里没有"简单删掉就好了"——你必须在某个地方记录"为什么要删"，否则3个月后没人记得为什么少了几行数据。这就是数据治理的意义。

---
## 🔬 概念群 2：Pandera — 运行时DataFrame Schema验证

> **关系类型**：[防护] — 主课讲"数据质量检查"（手动SQL），Pandera升级为"自动验证每条数据在进入分析前就符合预期"
> **[必修]** | 代码深度 | 来源：永久空白区 P1·数据质量工程 | 预计 15 分钟

### 🔴 认知冲突
你早上用 `dws_order_daily_1d` 表写日报，SQL跑了15分钟终于出结果——GMV 1.2亿。你发给了运营总监。
10分钟后运营总监截图回复："你这个GMV怎么比昨天跌了80%？"
你冲回去查SQL，发现 `dws_order_daily_1d` 表里今天的 `amount` 列数据类型被上游ETL任务改了——原本是 `DECIMAL(18,2)`，被某个"优化"改成了 `STRING`，
于是 `SUM(amount)` 对字符串做求和时发生了隐式转换……

**如果有一个工具在数据进入你的SQL之前就检查"amount是不是数字类型"，这个问题在凌晨5点就会被发现，而不是等你汇报给总监之后。**

### 📘 概念引入：Pandera
#### ① 一句话白话
"Pandera就是数据界的'海关安检'——每条数据进入你的分析前，必须先过X光机。"

#### ② 核心逻辑 + 具体例子
```python
import pandera as pa
schema = pa.DataFrameSchema({
    'order_id': pa.Column(str, pa.Check.str_length(8, 32), nullable=False),
    'amount': pa.Column(float, pa.Check.greater_than(0), nullable=False),
    'status': pa.Column(str, pa.Check.isin(['completed','cancelled','pending'])),
})
schema.validate(df)  # 不通过直接报错，不会静默产生错误结果
```

#### ③ 与主课核心概念的关系
主课区块2.3讲了"数据质量在流转中是如何损失的"，但只给了手动SQL检查模板。Pandera把这个过程自动化了——你不需要每天手动跑SQL检查，而是在读取数据时就自动验证。

### 🎯 在沙盘中的用武之地
快达出行的核心表 `dwd_order_detail_df` 每天早上5点产出。你在所有分析师的分析脚本开头加上Pandera验证：
- 如果amount列被错误地改成了STRING类型 → 脚本直接报错，不会产生错误日报
- 如果status出现了未预期的值（比如新增了'refunding'状态但维表没更新）→ 立即告警
- Schema定义本身就是给新人小陈的"数据字典"——看Schema就知道每列应该长什么样

In [2]:
# ===== 练习2：用Pandera构建数据质量安检门 =====
# 如果 pandera 未安装，先运行: !pip install pandera -q
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Check, Column, DataFrameSchema
from datetime import datetime

print('=' * 60)
print('Pandera 数据质量安检系统')
print('=' * 60)

# ===== 创建有隐蔽问题的模拟数据 =====
np.random.seed(42)
n = 1000

orders = pd.DataFrame({
    'order_id': [f'KD{2026}{i:07d}' for i in range(1, n+1)],
    'user_id': np.random.randint(10000, 20000, n),
    'driver_id': np.random.randint(5000, 8000, n),
    'amount': np.round(np.random.exponential(30, n), 2),
    'status': np.random.choice(['completed', 'cancelled', 'pending'], n, p=[0.7, 0.2, 0.1]),
    'city': np.random.choice(['北京', '上海', '深圳', '广州', '杭州'], n),
    'order_date': [datetime(2026, 7, 11) for _ in range(n)],
})

# 故意注入5个隐蔽问题
orders.loc[0, 'amount'] = -50.0    # 问题1: 负金额
orders.loc[1, 'amount'] = np.nan    # 问题2: NULL金额
orders.loc[2, 'status'] = 'deleted' # 问题3: 非法状态值
orders.loc[3, 'order_id'] = 'KD'    # 问题4: order_id太短
orders.loc[4, 'city'] = None        # 问题5: 城市为空

print(f'数据量: {len(orders)} 行, 注入了 5 个隐藏问题')

# ===== 定义 Schema =====
dwd_order_schema = DataFrameSchema(
    columns={
        'order_id': Column(str, Check.str_length(10, 30), nullable=False),
        'user_id': Column(int, Check.greater_than(0), nullable=False),
        'driver_id': Column(int, Check.greater_than(0), nullable=False),
        'amount': Column(float, Check.greater_than(0, error='金额必须为正数'), nullable=False),
        'status': Column(str, Check.isin(['completed','cancelled','pending','refunded']), nullable=False),
        'city': Column(str, nullable=True),
        'order_date': Column('datetime64[ns]', nullable=False),
    },
    strict=False,
)

# ===== 执行验证 =====
print('\n--- 执行Schema验证 ---')
try:
    dwd_order_schema.validate(orders, lazy=True)
    print('所有检查通过!')
except pa.errors.SchemaErrors as e:
    print(f'发现 {len(e.failure_cases)} 个问题:')
    print(e.failure_cases[['column', 'check', 'failure_case', 'index']].to_string())

# ===== 进阶：自动诊断与修复建议 =====
print('\n--- 问题诊断与修复建议 ---')

def diagnose_issues(df):
    report = []
    # 检查必要字段NULL率
    for col in ['order_id', 'user_id', 'amount', 'status']:
        null_rate = df[col].isnull().mean()
        if null_rate > 0:
            report.append(f'[{col}] NULL率={null_rate:.2%} → 回查上游ETL清洗逻辑')
    # 检查数值列异常值
    valid_amount = df['amount'].dropna()
    neg_rate = (valid_amount < 0).mean()
    if neg_rate > 0:
        report.append(f'[amount] 存在负金额({neg_rate:.2%}) → 暂停上游ETL,排查退款逻辑')
    # 检查分类字段
    expected_status = {'completed', 'cancelled', 'pending', 'refunded'}
    actual = set(df['status'].dropna().unique())
    unexpected = actual - expected_status
    if unexpected:
        report.append(f'[status] 非预期值: {unexpected} → 和业务方确认是否新增状态')
    return report

for issue in diagnose_issues(orders):
    print(f'  {issue}')

print('\n' + '='*60)
print('关键启示: Pandera让bug根本进不来——')
print('而不是发现bug后再修bug。')
print('='*60)

ModuleNotFoundError: No module named 'pandera'

### 🔧 练习反馈
| ✅ 参考答案 | ❓ 常见偏差 | 🔧 补救路径 |
|-----------|-----------|------------|
| Pandera发现5个问题(负金额/NULL/非法状态/ID太短/城市为空) | 只检查NULL忽略了值是否合法 | Schema要同时检查"有没有"和"对不对"
| 最严重的是负金额——可能表示上游ETL逻辑错误 | 认为"分析时过滤掉就行" | 如果是上游ETL错误，不只今天受影响——所有历史数据都坏了
| 用 `lazy=True` 一次性收集所有错误再报告 | 碰到第一个错误就停 | 批量排查需要看全局——lazy模式收集所有问题后一起展示

> 💡 记得用 `lazy=True` 参数——它会让Pandera收集所有错误后一次性报告（而不是碰到第一个错误就停），这对批量排查非常有用。

---
## 🔬 概念群 3：Great Expectations — 企业级数据质量自动化

> **关系类型**：[防护] — Pandera的"企业级表亲"。Pandera是代码内验证，GE是"数据质量平台"——自动生成HTML报告、支持调度、支持告警。
> **[必修]** | 代码深度 | 来源：永久空白区 P1·数据质量工程 | 预计 15 分钟

### 🔴 认知冲突
CTO张总在周会上问："上个月我们对外的12张报表全错了3天才修复。如果再发生一次，我怎么跟投资人解释？"
你说："我以后每天手动检查。"
张总摇头："你的人力和记忆力是有限的。我需要的是一个'如果我忘了检查，系统也不会忘'的方案。"

**这就是Great Expectations解决的问题——不是让你"更勤快"，而是让你"不需要勤快"。**

### 📘 概念引入：Great Expectations (GE)
#### ① 一句话白话
"Great Expectations就是你写一份'数据应该长什么样'的期望文档，它会自动帮你持续检查——像给数据上了24小时心电监护。"

#### ② 核心概念
- **Expectation（期望）**：一条具体的检查规则（如"order_id不能为NULL"）
- **Suite（期望套件）**：一组期望的集合，对应一张表
- **Checkpoint（检查点）**：在特定时间/事件触发检查
- **Data Docs（数据文档）**：自动生成的HTML报告

#### ③ 与Pandera的区别
| 维度 | Pandera | Great Expectations |
|------|---------|-------------------|
| 定位 | 代码内Schema验证 | 企业级数据质量平台 |
| 最佳场景 | 分析脚本第一步 | 数仓ETL后自动检查 |
| 报告 | 异常栈+文本 | 自动生成HTML可视化报告 |
| 调度 | 需要自己写 | 内置Checkpoint机制 |

### 🎯 在沙盘中的用武之地
快达出行每天早上8点产出运营日报(ads_operation_daily)。在调度系统中加入GE检查点：
- ETL任务完成后 → 自动触发GE期望套件 → 全部通过才允许下游报表生成
- 如果有人修改了上游表的字段口径 → GE检测到分布异常 → 阻断下游 → 发送钉钉告警
- 小陈入职时不再需要问"这张表的数据能信吗"——他只需要看Data Docs报告

In [ ]:
# ===== 练习3：用Great Expectations给快达出行建数据质量检查 =====
# 如果 great_expectations 未安装: !pip install great_expectations -q
import pandas as pd
import numpy as np

print('=' * 60)
print('Great Expectations: 企业级数据质量自动化')
print('=' * 60)

# ===== 准备测试数据 =====
np.random.seed(2026)
dates = pd.date_range('2026-07-01', '2026-07-11')
n_days = len(dates)

daily_summary = pd.DataFrame({
    'dt': dates,
    'total_orders': np.random.randint(7000000, 9000000, n_days),
    'total_gmv': np.round(np.random.uniform(5.0e8, 7.0e8, n_days), 2),
    'unique_users': np.random.randint(18000000, 22000000, n_days),
    'avg_order_value': np.round(np.random.uniform(65, 85, n_days), 2),
    'cancellation_rate': np.round(np.random.uniform(0.03, 0.08, n_days), 4),
})

# 注入问题：最后一天的数据有异常
daily_summary.loc[10, 'total_orders'] = 0        # 订单数为0!
daily_summary.loc[10, 'cancellation_rate'] = 1.5  # 取消率150%!

print('数据预览（最后3天，注意最后一行有异常）:')
print(daily_summary.tail(3).to_string())

# ===== 用检查清单方式定义期望 =====
print('\n--- 定义期望套件 ---')

checks = [
    ('日期列不为NULL', daily_summary['dt'].notnull().all()),
    ('订单数必须>0', (daily_summary['total_orders'] > 0).all()),
    ('取消率在0-1之间', daily_summary['cancellation_rate'].between(0, 1).all()),
    ('客单价在20-200之间', daily_summary['avg_order_value'].between(20, 200).all()),
    ('DAU在1500万-2500万之间', daily_summary['unique_users'].between(15_000_000, 25_000_000).all()),
]

# ===== 生成检查报告 =====
print('\n' + '='*60)
print('数据质量检查报告')
print(f'检查时间: 2026-07-12 08:00:00')
print(f'检查表: dws_order_daily_1d')
print('='*60)

passed = 0
failed_checks = []
for desc, result in checks:
    status = 'PASS' if result else 'FAIL'
    print(f'[{"✅" if result else "❌"} {status}] {desc}')
    if result:
        passed += 1
    else:
        failed_checks.append(desc)

print(f'\n结果: {passed}/{len(checks)} 通过, {len(failed_checks)} 失败')

if failed_checks:
    print(f'\n告警: 以下检查未通过:')
    for f in failed_checks:
        print(f'   - {f}')
    print(f'\n建议操作:')
    print(f'   1. 暂停 ads_ 层今天的数据产出')
    print(f'   2. 通知上游ETL责任人排查 dwd_order_detail_df')
    print(f'   3. 确认修复后重新触发DWS到ADS链路')

print('\n' + '='*60)
print('关键启示: GE的价值不在发现今天的数据有bug——')
print('而在昨天、前天、过去100天的数据都没bug,今天突然有bug。')
print('趋势的突然改变=真正的生产事故。')
print('='*60)

### 🔧 练习反馈
| ✅ 参考答案 | ❓ 常见偏差 | 🔧 补救路径 |
|-----------|-----------|------------|
| 检测到订单数为0和取消率1.5两个异常 | 认为"一眼就能看出来不需要GE" | 你能一眼看出今天的，那7天前的呢？GE的价值是"持续"不是"一次"
| GE应作为ETL调度的最后一个环节 | 把GE当作"分析师个人的检查工具" | GE应被集成到Airflow调度中——数据产出→自动检查→不通过则阻断下游
| GE最强使用方式不是手动跑——是把Checkpoint配置到调度系统里，数据产出后自动触发。

---
## 🔬 概念群 4：数据漂移检测 — 用统计方法发现数据"悄悄变了"

> **关系类型**：[防护] — 主课讲"数据质量检查"关注的是"数据有没有错"，漂移检测关注的是"数据有没有变"——两者互补
> **[必修]** | 代码深度 + 数学深度 | 来源：永久空白区 P1·数据漂移检测 | 预计 15 分钟

### 🔴 认知冲突
连续30天，你的日活数据都很稳定：19,500,000 ~ 19,900,000。数据质量检查全部通过——没有NULL，没有负值，类型都正确。
但今天DAU突然变成17,200,000（跌了13%）。

**"数据没出错" ≠ "数据没异常"。数据可以"悄悄地"改变它的分布模式，而传统的Schema验证对此完全盲视。**
这就是**数据漂移（Data Drift）**——数据的统计性质在时间中发生了变化。

### 📘 概念引入：数据漂移
#### ① 一句话白话
"你看一个人的体温——38.5°C是发烧。但如果你连续30天都是36.5°C，突然有一天变成37.8°C——格式上它只是一个数，统计上它就是一个信号。"

#### ② 三种漂移类型
- **协变量漂移（Covariate Shift）**：输入特征的分布变了。例：以前平均客单价65元，现在变成120元——用户群体变了
- **标签漂移（Label Shift）**：目标变量的分布变了。例：以前取消率3%，现在变成8%——政策/季节变了
- **概念漂移（Concept Drift）**：输入和输出的关系变了。例：以前发券→转化率+20%，现在发券→转化率+1%——用户麻木了

#### ③ 检测方法
- **KS检验**（Kolmogorov-Smirnov）：比较两个分布的差异，零假设是"来自同一分布"
- **PSI**（Population Stability Index）：银行业标准——PSI<0.1无变化, 0.1-0.25中等, >0.25显著
- **Wasserstein距离**：分布之间"搬运"的代价

In [ ]:
# ===== 练习4：实现数据漂移检测 =====
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('=' * 60)
print('数据漂移检测系统')
print('=' * 60)

# ===== 生成基准数据（过去30天）=====
np.random.seed(42)
baseline_dau = np.random.normal(19_500_000, 200_000, 30).astype(int)
baseline_aov = np.random.normal(72, 5, 30)

print(f'\n基准期 (过去30天):')
print(f'  DAU: 均值={baseline_dau.mean():,.0f}, 标准差={baseline_dau.std():,.0f}')
print(f'  客单价: 均值={baseline_aov.mean():.1f}, 标准差={baseline_aov.std():.1f}')

# ===== 方法1: KS检验 =====
def ks_drift_detection(baseline, current, alpha=0.01):
    statistic, p_value = stats.ks_2samp(baseline, current)
    has_drift = p_value < alpha
    return {'statistic': statistic, 'p_value': p_value, 'has_drift': has_drift}

# ===== 方法2: PSI =====
def calculate_psi(expected, actual, bins=10):
    all_data = np.concatenate([expected, actual])
    bin_edges = np.percentile(all_data, np.linspace(0, 100, bins + 1))
    expected_hist, _ = np.histogram(expected, bins=bin_edges)
    actual_hist, _ = np.histogram(actual, bins=bin_edges)
    expected_pct = np.where(expected_hist / len(expected) == 0, 0.0001, expected_hist / len(expected))
    actual_pct = np.where(actual_hist / len(actual) == 0, 0.0001, actual_hist / len(actual))
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

# ===== 实验 =====
print('\n--- 场景1: 正常波动(无漂移) ---')
today_normal = np.random.normal(19_500_000, 200_000, 1)[0]
r = ks_drift_detection(baseline_dau, [today_normal])
print(f'  当前DAU: {today_normal:,.0f}')
print(f'  KS p值: {r["p_value"]:.4f} → {"无漂移" if not r["has_drift"] else "有漂移!"}')

print('\n--- 场景2: DAU暴跌(有漂移) ---')
today_anomaly = 17_000_000
r = ks_drift_detection(baseline_dau, [today_anomaly])
print(f'  当前DAU: {today_anomaly:,.0f}')
print(f'  KS p值: {r["p_value"]:.6f} → {"有漂移!" if r["has_drift"] else "无漂移"}')

print('\n--- 场景3: 客单价结构漂移 ---')
baseline_aov_multi = np.random.normal(72, 5, 30)
actual_aov_multi = np.random.normal(100, 10, 30)
psi = calculate_psi(baseline_aov_multi, actual_aov_multi)
print(f'  基准均值: 72, 当前均值: 100')
print(f'  PSI: {psi:.4f}')
if psi < 0.1:
    print(f'  等级: 无显著变化')
elif psi < 0.25:
    print(f'  等级: 中等变化, 需要关注')
else:
    print(f'  等级: 显著变化, 需要告警!')

# ===== 综合监控报告 =====
print('\n' + '='*60)
print('快达出行 数据漂移监控报告')
print(f'生成时间: 2026-07-12 08:00')
print('='*60)

monitor_items = [
    ('dws_order_daily_1d.unique_users', baseline_dau, [today_normal], 'daily'),
    ('dws_order_daily_1d.cancellation_rate', np.random.beta(2, 50, 30), np.random.beta(5, 50, 1), 'daily'),
]

for metric_name, baseline, current, freq in monitor_items:
    psi_val = calculate_psi(np.array(baseline), np.array(current))
    ks_r = ks_drift_detection(np.array(baseline), np.array(current))
    if psi_val >= 0.25 or ks_r['has_drift']:
        status = 'ALERT'
    elif psi_val >= 0.1:
        status = 'WARN'
    else:
        status = 'OK'
    icon = {'ALERT': '🚨', 'WARN': '⚠️', 'OK': '✅'}[status]
    print(f'{icon} {status} | {metric_name} | PSI={psi_val:.4f} | KS p={ks_r["p_value"]:.4f}')

print('\n' + '='*60)
print('关键启示: Schema验证告诉你数据有没有格式错。')
print('漂移检测告诉你数据有没有内容变。两者缺一不可。')
print('='*60)

### 🔧 练习反馈
| ✅ 参考答案 | ❓ 常见偏差 | 🔧 补救路径 |
|-----------|-----------|------------|
| 场景2中KS检测到漂移(p极小)，场景3中PSI>0.25表示显著变化 | 认为只要格式正确就没问题 | 格式检查是Level1；分布检查是Level2；关系检查是Level3 |
| PSI和KS互补——PSI更适合离散变量，KS更适合连续变量 | 只用一个指标就下结论 | 多证据链一致时再告警——减少误报 |
| PSI阈值按指标重要性差异化（核心指标0.1，一般指标0.25）| 不管什么业务都用相同阈值 | 参考银行业的实践——差异化阈值 |

> 📖 想知道更多？搜索 "Evidently AI Python库"——它提供了开箱即用的漂移检测和HTML报告。

---
## 🔬 概念群 5：SQL反模式自动检测器

> **关系类型**：[防护] — 主课讲SQL"怎么写对"，这里讲"怎么避免写错"——一个自动扫描你的SQL找常见陷阱的工具
> **[必修]** | 代码深度 | 来源：永久空白区 P1·SQL反模式图鉴 | 预计 15 分钟

### 🔴 认知冲突
你写了这样一段SQL，跑了30分钟还没出结果：
```sql
SELECT *
FROM dwd_order_detail_df o
LEFT JOIN dim_user u ON o.user_id = u.user_id
WHERE o.dt = '2026-07-01'
  AND YEAR(o.order_date) = 2026
  AND u.city = '杭州'
LIMIT 100
```
你的同事看了一眼说："这个SQL有至少3个隐式陷阱——你发现了没？"

**问题在于**：① LEFT JOIN + WHERE u.city → 退化INNER JOIN；② YEAR()包裹分区键 → 分区裁剪失效；③ LIMIT无ORDER BY → 结果不确定；④ SELECT * → 表结构变化时下游代码崩溃。
**一个SQL可以有格式上的完美和逻辑上的陷阱——你需要的不是"写出更好的SQL"，而是"不被坏SQL坑到"。**

### 📘 概念引入：SQL反模式扫描器
#### ① 一句话白话
"这个工具就是SQL的'拼写检查'——不是帮你写出优美的SQL，而是阻止你提交有隐蔽bug的SQL。"

#### ② 检测的8种反模式
| 反模式 | 严重度 | 后果 |
|--------|--------|------|
| SELECT * | MEDIUM | 表结构变化时下游代码崩溃 |
| LEFT JOIN + WHERE右表条件 | HIGH | LEFT JOIN退化为INNER JOIN |
| 函数包裹分区键(YEAR/MONTH/SUBSTR) | HIGH | 分区裁剪失效，全表扫描 |
| 隐式类型转换 | LOW | 索引失效 |
| LIMIT 无 ORDER BY | MEDIUM | 每次返回不同结果 |
| 大表笛卡尔积(无ON条件) | CRITICAL | 行数爆炸 |
| COUNT(DISTINCT 多列) | MEDIUM | 大数据量下极慢 |
| NOT IN 子查询 | HIGH | NULL陷阱→结果为空集 |

#### ③ 与主课核心概念的关系
主课W4D1讲数据质量（数据本身有没有坏），反模式扫描器检查的是"你的SQL有没有静默的逻辑错误"——两者都是"防止你产生错误结论"的防线。

In [ ]:
# ===== 练习5：构建SQL反模式自动检测器 =====
import re
from dataclasses import dataclass
from typing import List

@dataclass
class AntiPattern:
    name: str
    severity: str  # CRITICAL, HIGH, MEDIUM, LOW
    line: int
    snippet: str
    explanation: str
    fix: str

class SQLAntiPatternScanner:
    def __init__(self):
        self.patterns = []

    def scan(self, sql: str) -> List[AntiPattern]:
        self.patterns = []
        lines = sql.split(chr(10))
        self._check_select_star(sql, lines)
        self._check_left_join_where(sql, lines)
        self._check_function_on_partition_key(sql, lines)
        self._check_implicit_conversion(sql, lines)
        self._check_limit_without_order(sql, lines)
        self._check_not_in_with_null(sql, lines)
        self._check_count_distinct_multiple(sql, lines)
        return self.patterns

    def _find_line(self, lines, pattern_text):
        for i, line in enumerate(lines):
            if pattern_text.strip()[:20] in line:
                return i + 1
        return 0

    def _check_select_star(self, sql, lines):
        if re.search(r'\bSELECT\s+\*', sql, re.IGNORECASE):
            self.patterns.append(AntiPattern(
                'SELECT *', 'MEDIUM', 1,
                'SELECT * FROM ...',
                '表结构变化时下游代码崩溃(新增/删除列)',
                '显式列出需要的列: SELECT order_id, amount, status FROM ...'))

    def _check_left_join_where(self, sql, lines):
        m = re.search(r'LEFT\s+JOIN\s+\w+\s+(\w+)\s+ON', sql, re.IGNORECASE)
        if m:
            alias = m.group(1)
            where_m = re.search(r'WHERE\s+(.+?)(?:GROUP|ORDER|LIMIT|$)', sql, re.IGNORECASE | re.DOTALL)
            if where_m and re.search(rf'\b{alias}\.', where_m.group(1)):
                self.patterns.append(AntiPattern(
                    'LEFT JOIN + WHERE右表条件', 'HIGH', self._find_line(lines, 'WHERE'),
                    f'WHERE ... {alias}.xxx ...',
                    f'WHERE过滤NULL值, LEFT JOIN退化为INNER JOIN',
                    f'将右表条件移到ON子句: ON o.id = {alias}.id AND {alias}.xxx = ...'))

    def _check_function_on_partition_key(self, sql, lines):
        for pattern, func_name in [
            (r'WHERE\s+.*YEAR\s*\(\s*(\w+)', 'YEAR()'),
            (r'WHERE\s+.*MONTH\s*\(\s*(\w+)', 'MONTH()'),
            (r'WHERE\s+.*DATE\s*\(\s*(\w+)', 'DATE()'),
        ]:
            m = re.search(pattern, sql, re.IGNORECASE)
            if m:
                col = m.group(1)
                self.patterns.append(AntiPattern(
                    f'分区键被函数包裹: {func_name}({col})', 'HIGH',
                    self._find_line(lines, func_name) or 1,
                    f'{func_name}({col})',
                    f'分区裁剪失效, 查询扫描所有分区',
                    f'改为范围查询: WHERE {col} >= ... AND {col} < ...'))

    def _check_implicit_conversion(self, sql, lines):
        m = re.search(r'WHERE\s+(\w+\.)?(\w+)\s*=\s*(\d+)', sql, re.IGNORECASE)
        if m:
            col = m.group(2)
            self.patterns.append(AntiPattern(
                '可能的隐式类型转换', 'LOW', self._find_line(lines, m.group(0)) or 1,
                m.group(0),
                f'如果{col}是字符串,和数字比较触发隐式转换→索引失效',
                f'显式转换: CAST({col} AS INT) = {m.group(3)}'))

    def _check_limit_without_order(self, sql, lines):
        if re.search(r'\bLIMIT\s+\d+', sql, re.IGNORECASE):
            if not re.search(r'\bORDER\s+BY\b', sql, re.IGNORECASE):
                self.patterns.append(AntiPattern(
                    'LIMIT 无 ORDER BY', 'MEDIUM', self._find_line(lines, 'LIMIT') or 1,
                    'LIMIT N (without ORDER BY)',
                    '返回不确定的结果——每次执行可能返回不同行',
                    '添加ORDER BY: ORDER BY o.order_date DESC LIMIT 100'))

    def _check_not_in_with_null(self, sql, lines):
        if re.search(r'\bNOT\s+IN\s*\(', sql, re.IGNORECASE):
            self.patterns.append(AntiPattern(
                'NOT IN 子查询的NULL陷阱', 'HIGH', self._find_line(lines, 'NOT IN') or 1,
                'NOT IN (subquery)',
                '子查询返回NULL→整个条件求值为FALSE→结果为空集',
                '改用 NOT EXISTS: WHERE NOT EXISTS (SELECT 1 FROM ... WHERE ...)'))

    def _check_count_distinct_multiple(self, sql, lines):
        if re.search(r'COUNT\s*\(\s*DISTINCT\s+\w+\s*,\s*\w+', sql, re.IGNORECASE):
            self.patterns.append(AntiPattern(
                'COUNT(DISTINCT 多列)', 'MEDIUM', 1,
                'COUNT(DISTINCT col1, col2)',
                '大数据量下极慢——Hive/Spark无法并行',
                '拆成子查询: SELECT COUNT(*) FROM (SELECT col1,col2 FROM t GROUP BY 1,2) sub'))

# ===== 测试 =====
print('=' * 60)
print('SQL反模式自动检测器')
print('=' * 60)

test_sql = '''
SELECT *
FROM dwd_order_detail_df o
LEFT JOIN dim_user u ON o.user_id = u.user_id
LEFT JOIN dim_driver d ON o.driver_id = d.driver_id
WHERE o.dt = '20260701'
  AND YEAR(o.order_date) = 2026
  AND u.city = '杭州'
  AND o.order_id NOT IN (
      SELECT order_id FROM tmp_blacklist
  )
LIMIT 100
'''

print('\n--- 待扫描SQL ---')
print(test_sql)

scanner = SQLAntiPatternScanner()
findings = scanner.scan(test_sql)

print(f'\n--- 扫描结果: 发现 {len(findings)} 个反模式 ---\n')
for i, f in enumerate(findings, 1):
    print(f'{i}. [{f.severity}] {f.name}')
    print(f'   ❌ {f.explanation}')
    print(f'   ✅ {f.fix}')
    print()

# 对比: 干净SQL
print('='*60)
print('对比: 扫描干净的SQL')
print('='*60)
clean_sql = '''
SELECT o.order_id, o.amount, o.status, u.city
FROM dwd_order_detail_df o
INNER JOIN dim_user u ON o.user_id = u.user_id
WHERE o.dt = '20260701'
ORDER BY o.order_date DESC
LIMIT 100
'''
clean_findings = scanner.scan(clean_sql)
if clean_findings:
    for f in clean_findings:
        print(f'  [{f.severity}] {f.name}')
else:
    print('  ✅ 未检测到反模式! 这是一个干净的SQL。')

print('\n' + '='*60)
print('关键启示: 集成到Git Hook或分析脚本的第一步,')
print('让坏SQL在提交前就被拦截。')
print('='*60)

### 🔧 练习反馈
| ✅ 参考答案 | ❓ 常见偏差 | 🔧 补救路径 |
|-----------|-----------|------------|
| 扫描器检测到6个反模式(SELECT *, LEFT JOIN+WHERE, YEAR(), 隐式转换, NOT IN, LIMIT无ORDER BY) | 觉得"能跑就行" | SQL "能跑"和"正确跑"之间有巨大鸿沟——WHERE过滤LEFT JOIN会让数据静默丢失 |
| NOT IN的NULL陷阱是CRITICAL级别的——它间歇性出错, 极难排查 | 从未遇到过所以不重视 | 可以先设成WARN级别, 但绝对不能关闭这个检查 |
> 💡 集成建议: 在Git pre-commit hook中运行扫描器; 在分析脚本的import之后立即检查SQL模板; CI/CD pipeline中自动扫描所有SQL文件。

---
## 🔬 概念群 6：数据血缘自动追踪——从SQL到依赖图

> **关系类型**：[落地] — 主课讲了血缘概念（一张静态图），这里讲"如何从SQL自动生成血缘"——把概念变成工具
> **[选修]** | 代码深度 | 预计 12 分钟

In [ ]:
# ===== 练习6：实现SQL表血缘解析器 =====
import re
from collections import defaultdict

print('=' * 60)
print('数据血缘自动追踪系统')
print('=' * 60)

class LineageParser:
    # 模拟快达出行的ETL链路
    ETL_SCRIPTS = {
        'task_dwd_order_detail': '''
            INSERT OVERWRITE TABLE dwd_order_detail_df PARTITION(dt='${yesterday}')
            SELECT o.order_id, o.amount, p.payment_method, t.distance
            FROM ods_order_log_df o
            LEFT JOIN ods_payment_log_df p ON o.order_id = p.order_id
            LEFT JOIN ods_trip_log_df t ON o.order_id = t.order_id
        ''',
        'task_dws_order_daily': '''
            INSERT OVERWRITE TABLE dws_order_daily_1d PARTITION(dt='${yesterday}')
            SELECT DATE(order_time) as dt, COUNT(DISTINCT order_id) as total_orders,
                   SUM(amount) as total_gmv, AVG(amount) as avg_order_value
            FROM dwd_order_detail_df WHERE dt = '${yesterday}'
            GROUP BY DATE(order_time)
        ''',
        'task_dws_user_daily': '''
            INSERT OVERWRITE TABLE dws_user_daily_1d PARTITION(dt='${yesterday}')
            SELECT u.user_id, COUNT(o.order_id) as order_count, SUM(o.amount) as total_spent
            FROM dim_user u
            LEFT JOIN dwd_order_detail_df o ON u.user_id = o.user_id
            WHERE o.dt = '${yesterday}' GROUP BY u.user_id
        ''',
        'task_ads_operation_daily': '''
            INSERT OVERWRITE TABLE ads_operation_daily PARTITION(dt='${yesterday}')
            SELECT o.dt, o.total_orders, o.total_gmv, u.total_spent
            FROM dws_order_daily_1d o
            JOIN dws_user_daily_1d u ON o.dt = u.dt
            WHERE o.dt = '${yesterday}'
        ''',
    }

    def parse_sources(self, sql):
        m = re.search(r'(?:INSERT\s+(?:OVERWRITE|INTO)\s+TABLE\s+)(\w+)', sql, re.IGNORECASE)
        target = m.group(1) if m else 'unknown'
        sources = re.findall(r'(?:FROM|JOIN)\s+(\w+)', sql, re.IGNORECASE)
        return [s for s in set(sources) if s != target]

    def parse_target(self, sql):
        m = re.search(r'(?:INSERT\s+(?:OVERWRITE|INTO)\s+TABLE\s+)(\w+)', sql, re.IGNORECASE)
        return m.group(1) if m else 'unknown'

    def build_lineage_graph(self):
        graph = defaultdict(list)
        for task_name, sql in self.ETL_SCRIPTS.items():
            sources = self.parse_sources(sql)
            target = self.parse_target(sql)
            for src in sources:
                graph[src].append(target)
        return dict(graph)

    def find_downstream_impact(self, graph, table_name, visited=None):
        if visited is None:
            visited = set()
        if table_name in visited:
            return set()
        visited.add(table_name)
        downstream = set()
        if table_name in graph:
            for target in graph[table_name]:
                downstream.add(target)
                downstream.update(self.find_downstream_impact(graph, target, visited))
        return downstream

    def find_upstream_dependencies(self, graph, table_name, visited=None):
        if visited is None:
            visited = set()
        if table_name in visited:
            return set()
        visited.add(table_name)
        upstream = set()
        for src, targets in graph.items():
            if table_name in targets:
                upstream.add(src)
                upstream.update(self.find_upstream_dependencies(graph, src, visited))
        return upstream

# ===== 运行 =====
parser = LineageParser()
graph = parser.build_lineage_graph()

print('\n--- 完整血缘图 ---')
for src, targets in sorted(graph.items()):
    for tgt in targets:
        print(f'  {src:30s} -> {tgt}')

# 场景1: 修改dwd_order_detail_df的amount字段口径——影响哪些下游?
print('\n' + '='*60)
print('场景: 修改 dwd_order_detail_df.amount 字段口径')
print('从含退款改为不含退款——影响哪些下游?')
print('='*60)
impact = parser.find_downstream_impact(graph, 'dwd_order_detail_df')
for table in sorted(impact):
    if table != 'dwd_order_detail_df':
        print(f'  ⚠️  {table}')
print(f'共计 {len(impact)-1} 个下游资产受影响')

# 场景2: ads_operation_daily今天为空——排查上游
print('\n' + '='*60)
print('场景: ads_operation_daily 今天为空——排查上游')
print('='*60)
upstream = parser.find_upstream_dependencies(graph, 'ads_operation_daily')
for table in sorted(upstream):
    print(f'  ← {table}')
print(f'共 {len(upstream)} 个上游依赖, 按依赖深度从小到大排查')

print('\n' + '='*60)
print('关键启示: 血缘图是数据治理的地图。')
print('没有地图, 修改字段口径就是在赌命。')
print('='*60)

### 🔧 练习反馈
| ✅ 参考答案 | ❓ 常见偏差 | 🔧 补救路径 |
|-----------|-----------|------------|
| 修改dwd_order_detail_df影响3个下游(dws_order_daily_1d, dws_user_daily_1d, ads_operation_daily) | 认为"改个字段而已" | 生产环境中一个字段变更可能影响几十张表——没有血缘就在赌博 |
| 排查从最上游(ods_)开始逐层往下 | 跳过中间层直接查ads | 中间层出问题→下游全错; ads层单独出问题→ads逻辑本身有问题 |
> 💡 如果想在生产环境实现真正的血缘追踪，推荐了解 dbt 的 `{{ ref() }}` 机制或 Apache Atlas。

---
## 🔬 概念群 7：数据延迟SLA监控自动化

> **关系类型**：[落地] — 主课给了一个手动排查清单，这里把它升级为自动监控+告警系统
> **[必修]** | 代码深度 | 预计 12 分钟

In [ ]:
# ===== 练习7：构建数据SLA监控系统 =====
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum

print('=' * 60)
print('数据SLA监控系统')
print('=' * 60)

class Severity(Enum):
    OK = '✅'
    WARN = '⚠️'
    CRITICAL = '🚨'

@dataclass
class TableSLA:
    table_name: str
    layer: str       # ODS/DWD/DWS/ADS
    expected_time: str  # '06:00'
    tolerance_min: int = 30
    upstream_depends: List[str] = field(default_factory=list)
    sla_owner: str = ''

@dataclass
class SLAStatus:
    table: TableSLA
    status: Severity
    delay_minutes: int
    current_status: str
    root_cause: Optional[str] = None
    estimated_recovery: Optional[str] = None

# 定义快达出行的SLA
sla_definitions = [
    TableSLA('ods_order_log_df', 'ODS', '03:00', 30, [], '数仓-老李'),
    TableSLA('ods_payment_log_df', 'ODS', '03:00', 30, [], '数仓-老李'),
    TableSLA('dwd_order_detail_df', 'DWD', '05:00', 60,
             ['ods_order_log_df', 'ods_payment_log_df'], '数仓-小明'),
    TableSLA('dws_order_daily_1d', 'DWS', '06:00', 60,
             ['dwd_order_detail_df'], '数仓-小明'),
    TableSLA('dws_user_daily_1d', 'DWS', '06:00', 60,
             ['dwd_order_detail_df', 'dim_user'], '数仓-小明'),
    TableSLA('ads_operation_daily', 'ADS', '08:00', 10,
             ['dws_order_daily_1d', 'dws_user_daily_1d'], '数仓-老李'),
    TableSLA('ads_ceo_dashboard', 'ADS', '08:00', 10,
             ['dws_order_daily_1d'], '数仓-老李'),
]

# 模拟真实状态（部分表延迟了）
now = datetime(2026, 7, 12, 8, 15)
table_status = {
    'ods_order_log_df':     {'produced_at': datetime(2026,7,12,3,5),  'status': 'produced'},
    'ods_payment_log_df':   {'produced_at': datetime(2026,7,12,3,45), 'status': 'produced'},
    'dwd_order_detail_df':  {'produced_at': datetime(2026,7,12,6,10), 'status': 'produced'},
    'dws_order_daily_1d':   {'produced_at': None, 'status': 'running'},
    'dws_user_daily_1d':    {'produced_at': None, 'status': 'failed'},
    'ads_operation_daily':  {'produced_at': None, 'status': 'not_started'},
    'ads_ceo_dashboard':    {'produced_at': None, 'status': 'not_started'},
}

def parse_time(time_str):
    h, m = map(int, time_str.split(':'))
    return now.replace(hour=h, minute=m, second=0, microsecond=0)

def check_sla(sla, current_time):
    status = table_status.get(sla.table_name, {})
    produced_at = status.get('produced_at')
    task_status = status.get('status', 'unknown')
    expected = parse_time(sla.expected_time)
    expected_with_tolerance = expected + timedelta(minutes=sla.tolerance_min)

    if produced_at:
        if produced_at <= expected_with_tolerance:
            delay = int((produced_at - expected).total_seconds() / 60)
            return SLAStatus(sla, Severity.OK, delay, task_status)
        else:
            delay = int((produced_at - expected).total_seconds() / 60)
            return SLAStatus(sla, Severity.WARN, delay, task_status,
                           root_cause='产出时间超过SLA容忍窗口')
    else:
        delay = int((current_time - expected).total_seconds() / 60)
        if task_status == 'failed':
            return SLAStatus(sla, Severity.CRITICAL, delay, task_status,
                           root_cause='ETL任务失败',
                           estimated_recovery='需要人工介入排查')
        elif task_status == 'running':
            upstream_cause = None
            for up in sla.upstream_depends:
                up_status = table_status.get(up, {})
                if up_status.get('status') in ('running', 'failed', 'not_started'):
                    upstream_cause = f'上游 {up} 状态异常({up_status.get("status")})'
                    break
            return SLAStatus(sla, Severity.WARN, delay, task_status,
                           root_cause=upstream_cause or 'ETL任务仍在运行中',
                           estimated_recovery='预计30分钟内产出')
        else:  # not_started
            upstream_cause = None
            for up in sla.upstream_depends:
                up_status = table_status.get(up, {})
                if up_status.get('status') != 'produced':
                    upstream_cause = f'等待上游 {up} 产出'
                    break
            return SLAStatus(sla, Severity.WARN, delay, task_status,
                           root_cause=upstream_cause or '调度尚未触发',
                           estimated_recovery='上游产出后1小时内')

# 运行SLA检查
print(f'\n检查时间: {now.strftime("%Y-%m-%d %H:%M")}')
print(f'数据日期: 2026-07-11 (T+1)\n')

results = [check_sla(sla, now) for sla in sla_definitions]
criticals = [r for r in results if r.status == Severity.CRITICAL]
warns = [r for r in results if r.status == Severity.WARN]
oks = [r for r in results if r.status == Severity.OK]

print('='*60)
print('SLA监控报告')
print('='*60)
print(f'{"层":<6} {"表名":<30} {"SLA":<8} {"状态":<8} {"延迟":<8} {"负责人"}')
print('-' * 85)
for r in results:
    delay_str = f'{r.delay_minutes}min' if r.delay_minutes > 0 else '准时'
    print(f'{r.table.layer:<6} {r.table.table_name[:28]:<30} {r.table.expected_time:<8} '
          f'{r.status.value:<8} {delay_str:<8} {r.table.sla_owner}')

print(f'\n汇总: {len(criticals)} CRITICAL | {len(warns)} WARN | {len(oks)} OK')

if criticals or warns:
    print('\n' + '='*60)
    print('自动告警 (可发送到钉钉/企业微信/Slack)')
    print('='*60)
    for r in criticals + warns:
        print(f'\n{r.status.value} {r.table.table_name}')
        print(f'  SLA: 每天 {r.table.expected_time} 产出')
        print(f'  当前状态: {r.current_status}')
        if r.root_cause:
            print(f'  根因: {r.root_cause}')
        if r.estimated_recovery:
            print(f'  预计恢复: {r.estimated_recovery}')
    # 推送给运营总监的简洁版
    print('\n--- 推送给运营总监(王姐)的简洁版 ---')
    print('王姐, 今天数据有部分延迟。')
    print('ads_operation_daily(运营日报)和ads_ceo_dashboard(CEO看板)尚未产出。')
    print('根因: 凌晨 ods_payment_log_df 延迟导致链路后推。')
    print('预计数据产出时间: 09:00, 会第一时间通知您。')

print('\n' + '='*60)
print('关键启示: 手动排查耗时间、易遗漏、无法提前预警。')
print('SLA监控让你从被运营催变成主动告知运营。')
print('='*60)

### 🔧 练习反馈
| ✅ 参考答案 | ❓ 常见偏差 | 🔧 补救路径 |
|-----------|-----------|------------|
| 1 CRITICAL(dws_user_daily_1d失败) + 3 WARN(dws_order_daily_1d运行中, ads*2未触发) | 以为ODS延迟不大—注意不到连锁效应 | 数据产出延迟有雪崩效应—ODS晚15分钟→DWD晚60分钟→DWS晚120分钟 |
| 根因在ods_payment_log_df晚了45分钟 | 只看最终结果不看根因 | SLA最有价值的信息不是"哪个表晚了"而是"第一个晚的表是哪个" |
> 💡 把SLA监控集成到Airflow的告警规则中——你需要的不是"任务失败告警"（太晚了），而是"SLA违反预警"（在失败之前就通知）。

---
## 🔬 概念群 8：数据架构演进——ETL→ELT→Zero-ETL & Lambda/Kappa

> **关系类型**：[替代] — 主课讲了ETL vs ELT（两种范式），这里补充"它们从哪来、往哪去、什么时候换"的决策框架
> **[选修]** | 案例深度 | 预计 12 分钟

### 📘 概念引入：架构选择的决策框架
#### ① 数据集成范式的三次进化
- **1980s-2000s: ETL** — "先洗干净再入库——因为存储很贵"，代表: Informatica, DataStage
- **2005-2020: ELT** — "先入库再洗——存储便宜了，先把数据保住"，代表: Hive, Spark SQL, dbt
- **2020-now: Zero-ETL** — "不用搬——直接在数据源上建虚拟视图"，代表: Trino/Presto, DuckDB

#### ② Lambda vs Kappa 架构
| 架构 | 思路 | 复杂度 | 适用 |
|------|------|--------|------|
| Lambda | 离线层(Hive)+实时层(Flink)两条链路 | 高(两套代码) | 数据准确性要求高 |
| Kappa | 只有实时层(Kafka做持久化) | 中(一套代码) | 实时性要求高 |
| Hybrid | 温数据ClickHouse+Doris, 冷数据Hive | 中 | 大部分互联网公司 |

#### ③ 选型决策树
```
需要秒级延迟?
├── 是 → 需要流式计算
│   └── 数据量>1TB/天?
│       ├── 是 → Lambda架构
│       └── 否 → Kappa架构
└── 否(T+1够用) → 传统离线数仓
    └── 有数仓工程师?
        ├── 是 → ELT + dbt
        └── 否 → DuckDB直查文件
```

### 🎯 在沙盘中的用武之地
> **场景**：CTO张总在规划下一年的技术roadmap，问你"我们应该投实时数仓还是优化离线数仓？"
>
> **你的分析**：
> 1. 盘点需求：日常分析(80%工作量)需要的是稳定性而非实时性；大促看板(每年3次)和风控(每天)需要实时
> 2. 量化ROI：离线优化（加DWS层预计算）每月省300小时分析师等待时间；实时数仓额外成本50万/年
> 3. 建议：**先优化离线，再按需引入实时**——不要为了"技术先进"而做实时数仓。CEO说"想要实时GMV"→ 用DuckDB直查MySQL只读副本即可，成本是Lambda的1/10。

> ⏸ **概念网络展开结束**。下面是第四部分——用综合沙盘演练验证你的迁移能力。

# 第四部分：综合沙盘演练
> ⏱ 预计 35 分钟 | 4道题 | 检验多概念协同和迁移能力

## 题1：多概念协同 —— 新数据源接入全流程
**场景**：快达出行新接入"企业用车"业务线。你作为Senior DA，需要设计从数据接入到质量监控的完整方案。要求至少用到本拓展中的4个概念。

请在下方code cell中写出你的完整方案。

In [ ]:
# ===== 综合沙盘 题1：企业用车数据接入方案 =====
print('=' * 60)
print('综合沙盘 题1: 企业用车数据接入方案')
print('=' * 60)

print('\n【1. CDC方案选型】')
print('方案: Log-based CDC (Canal)')
print('理由: 企业用车订单状态会变化(已预约→进行中→已完成→已结算),')
print('      必须捕获UPDATE事件; 日均5万单远小于主业务, Canal完全能支撑')
print('排除: Trigger-based CDC(拖慢业务库), Query-based(无法捕获DELETE)')

print('\n【2. Schema验证规则(Pandera)】')
rules = [
    ('enterprise_order_id', '格式ENT+日期+序号, 长度12-20, NOT NULL'),
    ('company_id', 'INT>0, NOT NULL, 外键关联dim_enterprise_company'),
    ('amount', 'DECIMAL>0, NOT NULL, <50000(单笔企业订单上限)'),
    ('status', 'ENUM: [booked,in_progress,completed,settled,cancelled]'),
    ('order_date', 'DATETIME, NOT NULL, 不能是未来日期'),
]
for col, rule in rules:
    print(f'  {col}: {rule}')

print('\n【3. SQL反模式防范】')
print('新人最可能犯的错:')
print('  ① LEFT JOIN dim_enterprise_company + WHERE company规模 → 退化INNER JOIN')
print('  ② YEAR(order_date)=2026 → 分区裁剪失效')
print('  ③ 两张表order_id重名不写别名 → 列歧义')
print('对策: SQL提交前跑反模式扫描器, 自动拦截')

print('\n【4. SLA定义】')
print('链路: MySQL→Canal→Kafka→ods_enterprise_order_log_df')
print('  → dwd_enterprise_order_detail_df → dws_enterprise_daily_1d')
print('SLA时间表:')
print('  ods: 03:00(容30min) → dwd: 05:30(容30min) → dws: 06:30(容30min)')
print('  纳入ads_operation_daily依赖: 07:30产出')

print('\n' + '='*60)
print('方案完整性检查: CDC选型+Pandera规则+反模式防范+SLA=完整方案')
print('='*60)

## 题2：沙盘变体 —— 规模翻倍后的架构调整
**场景**：快达出行收购了竞争对手，日均订单从800万翻倍到1600万，同时新增拼车和代驾两条业务线。
回答以下4个问题：

In [ ]:
# ===== 综合沙盘 题2：规模翻倍的架构调整 =====
print('=' * 60)
print('综合沙盘 题2: 1600万单/天的架构分析')
print('=' * 60)

print('\n【1. 离线数仓瓶颈】')
print('数据量翻倍: DWD清洗从1h→2h, DWS聚合从1.5h→3h')
print('最大瓶颈: DWS层多表JOIN+GROUP BY')
print('建议: 短期Spark换Hive(快2-3倍), 中期DWS迁ClickHouse(快10-50倍)')

print('\n【2. Kafka削峰能力】')
print('双十一峰值=日常10倍=10,000 msg/s')
print('对策: Partition扩容到100-200, 消息保留时间从7天缩到3天')

print('\n【3. 质量监控调整】')
print('新增: 拼车订单重复率检查, 代驾距离异常检测(>100km告警)')
print('阈值更新: 行数波动率从20%放宽到30%(业务快速增长期)')

print('\n【4. SLA时间表调整】')
print('原SLA: ods 03:00 → dwd 05:00 → dws 06:00 → ads 08:00')
print('新SLA: ods 03:00 → dwd 06:00 → dws 07:30 → ads 09:00')
print('但09:00太晚! 解决: ads层迁ClickHouse, 聚合从30min→1-2min, 维持08:00产出')

print('\n结论: 架构不是一成不变的。规模翻倍→架构必须重新评估。')
print('='*60)

## 题3：边界判断 —— 什么时候不该用这些工具？
快速判断以下3个场景中，你是"用上了"还是"过度工程"了？

In [ ]:
# ===== 综合沙盘 题3：边界判断 =====
print('=' * 60)
print('综合沙盘 题3: 边界判断')
print('=' * 60)

scenarios = [
    ('A', '5人创业团队, 每天50单, 没有数仓', 'Great Expectations',
     '不该用(过度工程)',
     '替代: Pandas .describe() + .isnull().sum() 手动检查即可。'),
    ('B', '200人数据团队, 每天500万单, T+1链路', 'Pandera',
     '该用(基础防护)',
     '每个分析脚本第一步都应有Schema验证+SQL反模式扫描。'),
    ('C', '20人团队, CEO想要实时GMV', 'Lambda架构',
     '谨慎——可能不需要全套Lambda',
     '用Flink消费Kafka写Redis→看板读Redis, 成本是Lambda的1/10。'),
]
for sid, name, tool, verdict, reason in scenarios:
    print(f'\n场景{sid}: {name}')
    print(f'  工具: {tool}')
    print(f'  判断: {verdict}')
    print(f'  理由: {reason}')

print('\n' + '='*60)
print('核心原则: 规模决定工具。5人→Pandas够, 50人→Pandera, 500人→GE全家桶。')
print('不要让工具比问题更大。')
print('='*60)

## 题4：落地 —— 数据质量体系建设提案
**场景**：CTO要求你下周在周会上提案——"建立数据质量体系"。写一份执行摘要。

In [ ]:
# ===== 综合沙盘 题4：数据质量体系建设提案 =====
print('=' * 60)
print('综合沙盘 题4: 数据质量体系建设 - 执行摘要')
print('=' * 60)

proposal = '''
┌─────────────────────────────────────────────────────────────┐
│  数据质量体系建设提案                                        │
│  提案人: 高级数据分析师 | 日期: 2026-07-14 | 提交: CTO张总   │
└─────────────────────────────────────────────────────────────┘

【1. 问题量化】过去3个月: 7次数据事故
  3次ETL失败→延迟, 2次口径变更→报表错误,
  1次Kafka故障→数据丢失, 1次SQL反模式→错误结论
  影响: 累计浪费120小时, 运营信任度下降

【2. 三步走方案 (0元预算, 纯开源工具)】
  Phase 1(第1月): Pandera + SQL反模式扫描器 (1个分析师50%时间)
  Phase 2(第2-3月): Great Expectations + SLA监控 (+0.5个数仓工程师)
  Phase 3(第4-6月): 数据漂移检测 + 血缘自动化 (+1个数仓工程师)

【3. ROI】投入4.5人月 → 事故减少71% → 回收期<1季度

【4. 需要的支持】分析师50%时间x6个月 + 数仓工程师配合 + Airflow权限

【5. 第一个月交付物】
  W1: 数据质量基线报告 + 5张核心表Pandera Schema
  W2: SQL反模式扫描器(Git Hook) + GE期望套件
  W3: GE Data Docs质量看板 + SLA监控配置
  W4: 周会演示 + 第一期质量报告
─────────────────────────────────────────────────────────────
  一句话: 0元预算+1个分析师50%时间, 3个月把数据质量
  从出事了再救火升级为出事前就能发现。
─────────────────────────────────────────────────────────────
'''
print(proposal)
print('='*60)

> ⏸ **综合沙盘演练结束**。下面是第五部分——知识网络全景图。

# 第五部分：知识网络全景图

## 📡 概念关系全景表

| 概念 | 关系类型 | 关联概念 | 协同使用 |
|------|---------|---------|---------|
| CDC实战模拟 | [前置] — 理解数据如何移动 | → 数据血缘、SLA监控 | CDC保证数据到达 + SLA监控保证按时到达 |
| Pandera Schema验证 | [防护] — 格式级防线 | → GE、SQL反模式 | Pandera做第一道防线(格式)，GE做第二道(业务规则) |
| Great Expectations | [防护] — 业务规则级防线 | → Pandera、漂移检测 | GE检查今天的数据是否合规，漂移检测对比历史 |
| 漂移检测 | [防护] — 分布级防线 | → GE、SLA监控 | 分布变了但格式没变→只有漂移检测能发现 |
| SQL反模式检测 | [防护] — 代码级防线 | → Pandera、数据血缘 | SQL写对 + 数据没坏 = 分析结果可信 |
| 数据血缘追踪 | [落地] — 影响分析 | → SLA监控、GE | 上游变了→血缘告诉你要检查哪些下游→自动验证 |
| SLA监控 | [落地] — 时效性保障 | → 数据血缘、CDC | 血缘告诉你上游→SLA监控上游→延迟自动告警 |
| 架构演进决策 | [替代] — 场景适配 | → 全部概念 | 小团队vs大团队，选不同的工具组合 |

## 🔗 关系图解

```
         数据流转基础(CDC)
              │
    ┌─────────┼─────────┐
    │         │         │
    ▼         ▼         ▼
 数据质量    数据血缘   时效保障
    │         │         │
    ├─Pandera │         ├─SLA监控
    ├─GE      │         └─架构演进
    ├─漂移    │
    └─反模式  │
    │         │
    └────┬────┘
         │
         ▼
     分析结果可信
```

## 📊 三版本覆盖对比

| 主题维度 | 主课程 (W4D1) | 本拓展 | 后续课程 |
|---------|-------------|--------|---------|
| 数据如何从OLTP到OLAP | ✅ 概念级 | ✅ CDC工程实现 | - |
| 数据质量问题 | ✅ 手动SQL检查 | ✅ 自动化三件套(Pandera+GE+漂移) | W4D3部分涉及 |
| SQL质量 | - | ✅ 反模式自动检测 | W5-W6部分涉及 |
| 数据血缘 | ✅ 概念+示例 | ✅ 自动解析+依赖图 | - |
| 数据延迟 | ✅ 手动排查清单 | ✅ SLA自动监控+告警 | - |
| 架构决策 | ✅ ETL vs ELT | ✅ 三范式演进+Lambda/Kappa | W4D4-D5浅层涉及 |
| 数据质量工程化 | ❌ 完全空白 | ✅ Pandera+GE+漂移完整实践 | - |

## 📚 延伸阅读

1. **Pandera官方文档**: pandera.readthedocs.io — Schema定义的完整指南
2. **Great Expectations官方教程**: docs.greatexpectations.io — Getting Started tutorial
3. **Evidently AI**: github.com/evidentlyai/evidently — 开箱即用的漂移检测+HTML报告
4. **dbt官方文档**: docs.getdbt.com — 数据分析工程化的行业标准
5. **《Designing Data-Intensive Applications》** 第3-4章 — 存储与编码的底层原理
6. **《Fundamentals of Data Engineering》** 第5-7章 — 数据生成、存储、摄取
7. **Apache Kafka官方文档**: Design章节 — 理解Exactly-Once语义
8. **Flink CDC官方文档**: ververica.github.io/flink-cdc-connectors

## 🔄 跨模块连接

| 连接方向 | 关联内容 | 连接点 |
|---------|---------|--------|
| ← W1D2 指标工具箱 | 指标定义规范 | 本拓展的Pandera/GE负责验证指标数据本身是否可靠 |
| ← W1D6 异动分析 | 异动排查三步法 | 排查前先确认数据质量→避免数据问题当业务异动 |
| ← W3D1 归因模型 | ISS框架 | 归因分析的前提是被归因的数据是准确的 |
| → W4D2 数仓建模 | DWD/DWS层设计 | 本拓展的血缘追踪帮你理解建模的为什么 |
| → W4D4 Hive实战 | 分区/桶 | 反模式检测器中的函数包裹分区键会在W4D4深入 |
| → W5-W6 SQL进阶 | 复杂SQL | 反模式检测器是你写出高效SQL的安全带 |
| → W11 综合项目 | 端到端分析 | 数据质量自动化是你项目工程化程度的关键差异点 |

# 第六部分：元认知自评
> ⏱ 预计 15 分钟 | 诚实自评——发现自己不知道什么比假装知道更有价值

## 📊 学习循环自评表

| 概念群 | 没理解(0) | 能看懂(1) | 能改代码(2) | 能给同事讲(3) |
|--------|----------|----------|-----------|-------------|
| 1. CDC实战模拟 | | | | |
| 2. Pandera Schema验证 | | | | |
| 3. Great Expectations | | | | |
| 4. 数据漂移检测 | | | | |
| 5. SQL反模式检测 | | | | |
| 6. 数据血缘追踪 | | | | |
| 7. SLA监控自动化 | | | | |
| 8. 架构演进决策 | | | | |

**计分**: 0-8分→重新过一遍 / 9-16分→重点强化薄弱概念 / 17-24分→进入综合沙盘 / 24分+→进入下一步

## 🧠 多维度自评量表 (每题0-2分)

| 维度 | 0分: 做不到 | 1分: 能做但需查资料 | 2分: 能独立完成 | 自评 |
|------|-----------|-------------------|---------------|------|
| **概念理解** | 说不清CDC/Pandera/漂移是什么 | 能说定义但讲不清为什么 | 能用自己的话解释为什么需要和怎么选 | |
| **代码实现** | 看了代码不理解 | 能运行但不会改 | 能根据自己业务修改Pandera Schema/SQL扫描规则 | |
| **沙盘应用** | 不知道在快达出行中怎么用 | 大概知道场景但对不上具体表 | 能把每个概念映射到具体的表和SLA | |
| **架构决策** | 只能用先进判断工具好坏 | 能分辨场景但说不清原因 | 能根据团队规模/数据量选择最合适的工具组合 | |
| **迁移能力** | 换一个行业就不知道怎么用 | 知道概念通用但不知怎么调参数 | 能识别新场景中的数据质量风险点并提出工具选择 | |
| **业务沟通** | 无法向运营解释数据延迟的原因 | 能解释但给不出时间预期 | 能在延迟时主动通知并给出准确的恢复时间 | |

**总分: __ / 12**
- 0-4分: 🔴 重点突破概念群1-4(基础防护)——数据质量工程师的基本盘
- 5-8分: 🟡 能理解和运行代码, 但独立应用还有距离——去沙盘里把每个概念都跑一遍
- 9-10分: 🟢 概念已内化——在你的实际工作中挑一张表写Pandera Schema + GE期望
- 11-12分: ⭐ 你已具备数据工程思维——可以考虑深入研究dbt或Apache Atlas

## 🎯 薄弱点精准强化方案

**如果0-1分集中在代码实现维度**: 回去把概念群2-7的代码cell全部删掉, 从零重写一遍（不要复制粘贴）。用你自己的表名和字段名。
**如果0-1分集中在架构决策维度**: 阅读延伸阅读#5《Designing Data-Intensive Applications》第3章（存储与检索）——这不是代码书, 它教你为什么系统被设计成这样。
**如果0-1分集中在业务沟通维度**: 练习——用运营总监能听懂的话解释数据SLA监控。不要说KS检验的p值, 说系统会自动在数据延迟时发消息告诉你大概多久能好。

## 📝 沙盘复述测试
**不要看笔记**, 用自己的话回答以下3个问题：
1. 快达出行的数据质量体系分为哪三层防线？每层解决什么问题？
2. 如果dwd_order_detail_df的amount字段口径变了, 你需要检查哪些东西？
3. CTO说我想做实时数仓, 你的分析框架是什么？

**复述后对照**:
- 问题1: 格式层(Pandera)→业务规则层(GE)→分布层(漂移检测PSI/KS)
- 问题2: ①血缘追踪找所有下游→②GE验证所有下游表→③SQL反模式扫描→④SLA监控确认产出→⑤通知下游owner
- 问题3: ①明确需求(什么场景？什么指标？)→②量化成本(实时比离线贵3-10倍)→③按需选择(一个指标实时≠整个数仓实时)→④如果不能解释为什么需要实时——说明不需要

## ✅ 循环毕业检查清单
- [ ] 我能向Junior分析师解释什么是CDC, 以及为什么数据从MySQL到数仓不是复制粘贴那么简单
- [ ] 我能为一张数仓表写出完整的Pandera Schema (5+列, 含类型/非空/值域约束)
- [ ] 我能用Great Expectations为一个ETL链路配置期望套件
- [ ] 我能用PSI和KS检验判断数据今天的分布和昨天是否一样
- [ ] 我能用SQL反模式扫描器找出>3个SQL中的问题
- [ ] 我能从SQL中解析出表依赖关系
- [ ] 我能为5张核心表定义SLA, 并写出违反SLA时的告警消息
- [ ] 我能根据团队规模和数据量推荐合适的工具组合
- [ ] 我能向运营总监解释为什么数据延迟了以及什么时候能好——用她听得懂的话

**全部勾完 → 🎉 你已具备数据工程思维——去把你公司的数据质量体系建起来吧！**

---

## 📌 拓展小结

> **本拓展为W4D1"数据从哪来"补充了8个课程未覆盖的工程概念——从CDC的底层一致性问题，到Pandera/GE/漂移检测的三层数据质量防护，再到SQL反模式自动检测、数据血缘追踪、SLA监控，以及架构演进决策。核心信息只有一个：分析师不只是"用数据的人"——你是"让数据变得可信、可追溯、可监控的人"。从今天开始，你写的每一个SQL前面都应该有一个Pandera验证。你排查的每一个"数据对不上"都应该有一张血缘图。你管理的每一个ETL链路都应该有一个SLA。这就是从"取数工具人"到"数据建设者"的跃迁路径。**

---

*03analyst_thinking · Week 4 Day 1 拓展 · 数据工程思维深度拓展*
*生成工具: analyst-expand v4.1 · 生成日期: 2026-07-12*